# Doubles Pairing Analysis

This notebook loads the registration sheet, keeps players who joined doubles, and pairs the highest level player with the lowest level player to keep team averages close to the group average.

In [ ]:
from pathlib import Path

import pandas as pd

file_path = Path("files") / "Social Tennis Tournament Registration Aug 15 (Responses).xlsx"

name_col = "Name"
game_col = "Which game would you like to join?"
level_col = "What is your tennis level?"

df = pd.read_excel(file_path)

doubles = (
    df.loc[df[game_col].fillna("").str.contains("Doubles", case=False), [name_col, level_col]]
    .rename(columns={name_col: "name", level_col: "level"})
    .assign(name=lambda x: x["name"].astype(str).str.strip())
    .dropna(subset=["name", "level"])
    .sort_values("level", ascending=False)
    .reset_index(drop=True)
)

group_avg = doubles["level"].mean()

print(f"Doubles players: {len(doubles)}")
print(f"Average doubles level: {group_avg:.2f}")
doubles

In [ ]:
if len(doubles) % 2 != 0:
    raise ValueError("The number of doubles players must be even to make complete pairs.")

top_half = doubles.iloc[: len(doubles) // 2].reset_index(drop=True)
bottom_half = doubles.iloc[len(doubles) // 2 :].sort_values("level").reset_index(drop=True)

pairs = pd.DataFrame(
    {
        "player_1": top_half["name"],
        "level_1": top_half["level"],
        "player_2": bottom_half["name"],
        "level_2": bottom_half["level"],
    }
)

pairs["team_avg"] = pairs[["level_1", "level_2"]].mean(axis=1)
pairs["distance_from_group_avg"] = (pairs["team_avg"] - group_avg).abs().round(2)
pairs = pairs.sort_values(["distance_from_group_avg", "team_avg"]).reset_index(drop=True)

pairs

In [ ]:
pairs[["player_1", "player_2", "level_1", "level_2", "team_avg", "distance_from_group_avg"]]